# Treinamento e avaliação no conjunto de testes com RF-DETR

Este notebook reproduz o fluxo das variantes RF-DETR Nano e Medium para o dataset Urban Disaster Monitor. Ele treina a variante escolhida e a avalia no **conjunto de testes**.

A API de treinamento do RF-DETR requer uma exportação COCO com as pastas `train`, `valid` e `test`, cada uma contendo `_annotations.coco.json`. Baixe o dataset no formato COCO pelo [Roboflow Universe](https://universe.roboflow.com/ufrnprojects-xlut9/urban-disaster-monitor/dataset/4) antes de executar as células de treinamento.

Os parâmetros abaixo correspondem aos experimentos registrados: 50 épocas, *batch size* 4, acumulação de gradientes 4 e resolução de 384 px (Nano) ou 576 px (Medium).

## 1. Dependências e GPU

Execute este notebook em um ambiente Python 3.10+ e, preferencialmente, com GPU CUDA.

In [ ]:
!pip install "rfdetr[train]>=1.8.3" pandas --quiet

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. Validação do dataset

Defina `DATASET_DIR` como a raiz da exportação COCO. A validação abaixo evita iniciar um treino usando por engano a exportação YOLO armazenada no repositório.

In [ ]:
from pathlib import Path

DATASET_DIR = Path("/content/dataset")  # Altere se a exportação COCO estiver em outro local
required_annotations = [DATASET_DIR / split / "_annotations.coco.json" for split in ("train", "valid", "test")]
missing = [path for path in required_annotations if not path.exists()]
if missing:
    missing_paths = "\n".join(f"- {path}" for path in missing)
    raise FileNotFoundError(
        "Uma exportação COCO é obrigatória. Arquivos de anotação ausentes:\n" + missing_paths
    )

print(f"Using COCO dataset: {DATASET_DIR.resolve()}")

## 3. Configurar e treinar

Escolha `nano` ou `medium`. `NUM_CLASSES = 7` preserva a configuração de classes registrada nos artefatos atuais do RF-DETR (`objects` mais as seis classes do projeto). Defina `RUN_TEST = True` para gerar métricas no conjunto de testes ao término do treino.

In [ ]:
from rfdetr import RFDETRMedium, RFDETRNano

VARIANT = "medium"  # "nano" ou "medium"
NUM_CLASSES = 7
EPOCHS = 50
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 1e-4
RESOLUTION = {"nano": 384, "medium": 576}[VARIANT]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("runs") / f"rfdetr-{VARIANT}-urban-disaster"
RUN_TEST = True

model_class = {"nano": RFDETRNano, "medium": RFDETRMedium}[VARIANT]
model = model_class(num_classes=NUM_CLASSES)

model.train(
    dataset_dir=str(DATASET_DIR),
    output_dir=str(OUTPUT_DIR),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LEARNING_RATE,
    resolution=RESOLUTION,
    device=DEVICE,
    eval_interval=1,
    run_test=RUN_TEST,
)

## 4. Inspecionar métricas registradas

O RF-DETR grava a configuração e as métricas na pasta da execução. Mantenha esses arquivos junto ao checkpoint para comparar os resultados com a avaliação de YOLOv26m no conjunto de testes.

In [ ]:
import pandas as pd

metrics_path = OUTPUT_DIR / "metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    metric_columns = [
        column for column in ("epoch", "val/mAP_50", "val/mAP_50_95", "val/precision", "val/recall", "val/F1")
        if column in metrics.columns
    ]
    display(metrics[metric_columns].dropna(how="all").tail())
else:
    print(f"Execute o treinamento primeiro; métricas não encontradas em {metrics_path}")

for artifact in ("training_config.json", "results.json", "checkpoint_best_total.pth"):
    print(f"{artifact}: {(OUTPUT_DIR / artifact).exists()}")

## 5. Inferência com o melhor checkpoint

Use uma imagem do conjunto de testes para uma verificação qualitativa. O checkpoint treinado também pode ser enviado ao repositório do projeto no Hugging Face para uso pelo app Gradio.

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".webp"}
test_images = sorted(path for path in (DATASET_DIR / "test").iterdir() if path.suffix.lower() in image_extensions)
if not test_images:
    raise FileNotFoundError("Nenhuma imagem de teste foi encontrada ao lado de test/_annotations.coco.json.")

checkpoint_path = OUTPUT_DIR / "checkpoint_best_total.pth"
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Melhor checkpoint não encontrado: {checkpoint_path}")

best_model = model_class(num_classes=NUM_CLASSES, pretrain_weights=str(checkpoint_path))
detections = best_model.predict(str(test_images[0]), threshold=0.30)
print(f"Imagem de teste: {test_images[0].name}")
print(f"Detecções: {len(detections.xyxy)}")
detections